In [1]:
import pandas as pd
import numpy as np
import os
import sys
# Add the 'src' directory to the system path
src_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
sys.path.insert(0, src_dir)

# Now you can import the module
import models_utils

# from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import ElasticNet
from xgboost import XGBRegressor


from joblib import dump, load
import itertools

In [2]:
transcriptomics = load("data/transcriptomics_matched.joblib")
proteomics = load("data/proteomics_matched.joblib")

In [3]:
indices = proteomics.index

## Elastic Net

In [4]:
elasticnet_model = ElasticNet()

In [6]:
param_grid_elasticnet = {
    'alpha': [0.1, 0.3],  # Regularization strength
    'l1_ratio': [0.1, 0.5, 0.9] #,  # Mix of L1 and L2 regularization
    #'max_iter': [1000, 5000, 10000],          # Number of iterations
    #'tol': [1e-4, 1e-3, 1e-2],                # Tolerance for optimization
    #'selection': ['cyclic', 'random'],        # Update rule for coordinate descent
}

In [8]:
# Use the full adaptive pipeline with ElasticNet
elasticnet_scores = models_utils.full_adaptive_pipeline(
    model = elasticnet_model,
    param_grid = param_grid_elasticnet,
    X = transcriptomics,
    y = proteomics,
    anova_features = 20,
    n_samples = 1,
    n_proteins = 1
)

In [9]:
elasticnet_scores

,protein,LOO sample,true,predicted,params
0,A2M,MDAMB468_BREAST,-1.243308,-0.315027,"{'alpha': 0.3, 'l1_ratio': 0.1}"


In [7]:
elasticnet_scores

,protein,LOO sample,true,predicted,params
0,A2M,MDAMB468_BREAST,-1.517692,-0.049315,"{'alpha': 0.1, 'l1_ratio': 0.7, 'max_iter': 10..."
1,A2M,SH4_SKIN,2.690929,0.425941,"{'alpha': 1, 'l1_ratio': 0.1, 'max_iter': 1000..."
2,A2M,AU565_BREAST,-0.608218,-0.748010,"{'alpha': 1, 'l1_ratio': 0.1, 'max_iter': 1000..."
3,A2M,KMRC1_KIDNEY,-0.205904,1.069279,"{'alpha': 1, 'l1_ratio': 0.0, 'max_iter': 5000..."
4,A2M,CAL51_BREAST,1.057574,0.577427,"{'alpha': 1, 'l1_ratio': 0.1, 'max_iter': 1000..."
5,A2M,RPMI7951_SKIN,-0.618660,-1.046130,"{'alpha': 1, 'l1_ratio': 0.0, 'max_iter': 1000..."
6,A2M,RERFLCMS_LUNG,-1.805030,-0.868340,"{'alpha': 1, 'l1_ratio': 0.1, 'max_iter': 1000..."
7,A2M,IGR37_SKIN,-0.529690,-0.524373,"{'alpha': 1, 'l1_ratio': 0.1, 'max_iter': 1000..."
8,A2M,VMRCRCW_KIDNEY,0.008330,-0.355430,"{'alpha': 0.1, 'l1_ratio': 1.0, 'max_iter': 10..."
9,A2M,HEP3B217_LIVER,0.770297,0.273754,"{'alpha': 1, 'l1_ratio': 0.1, 'max_iter': 1000..."


## XGBoost

In [11]:
xgb = XGBRegressor(device='cpu', tree_method='hist')

In [15]:
param_grid_xgboost = {
    'eta': [0.03],         # Learning rate (smaller values for better performance)
    'max_depth': [3, 6],            # Maximum tree depth
    #'subsample': [0.5, 0.7, 0.9, 1.0],     # Fraction of samples for training each tree
    'colsample_bytree': [0.5, 1.0],  # Fraction of features for each tree
    #'gamma': [0, 0.1, 0.5, 1],             # Minimum loss reduction to make a further partition
    #'min_child_weight': [1, 3, 5, 7],      # Minimum sum of instance weight (Hessian) needed in a child
    #'lambda': [0, 1, 5, 10],               # L2 regularization term
    #'alpha': [0, 0.1, 0.5, 1],             # L1 regularization term
    #'n_estimators': [100, 200, 500],       # Number of boosting rounds
}

In [ ]:
xgb_scores = models_utils.full_adaptive_pipeline(
    model = xgb,
    param_grid = param_grid_xgboost,
    X = transcriptomics,
    y = proteomics,
    anova_features = 20,
    n_proteins = 1,
    n_samples = 1
)

KeyboardInterrupt: 

In [17]:
xgb_scores

,protein,LOO sample,true,predicted,params
0,A2M,MDAMB468_BREAST,-0.878405,-0.865976,"{'eta': 0.03, 'max_depth': 3, 'colsample_bytre..."
